# Chapter 2 TA Session: Root Finding with Bisection

In this introduction, we implement the **bisection method** to approximate a root of a continuous function on an interval $[a,b]$ where $f(a)f(b) < 0$.

At each iteration, we track:

- midpoint approximation $p_n$
- function value $f(p_n)$
- absolute step error $|p_n - p_{n-1}|$
- residual error $|f(p_n)|$

In [1]:
import math
from typing import Callable


def bisection_with_errors(
    f: Callable[[float], float],
    a: float,
    b: float,
    tol: float = 1e-8,
    max_iter: int = 100,
):
    """Return bisection root estimate and per-iteration error data."""
    fa, fb = f(a), f(b)
    if fa * fb >= 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")

    rows = []
    prev_p = None

    for n in range(1, max_iter + 1):
        p = (a + b) / 2.0
        fp = f(p)

        step_error = abs(p - prev_p) if prev_p is not None else float("nan")
        residual_error = abs(fp)

        rows.append(
            {
                "n": n,
                "a": a,
                "b": b,
                "p_n": p,
                "f(p_n)": fp,
                "|p_n - p_(n-1)|": step_error,
                "|f(p_n)|": residual_error,
            }
        )

        if residual_error < tol or (prev_p is not None and step_error < tol):
            return p, rows

        if fa * fp < 0:
            b, fb = p, fp
        else:
            a, fa = p, fp

        prev_p = p

    return p, rows

In [4]:
# Example: solve x^2 - cos(x) = 0 on [0, 1]
f = lambda x: x**2 - math.cos(x)

root, history = bisection_with_errors(f, a=0.0, b=1.0, tol=1e-10, max_iter=60)

print(f"Approximate root: {root:.12f}")
print(f"f(root): {f(root):.3e}")
print("\nIteration errors:")
print("n   p_n            |p_n-p_(n-1)|    |f(p_n)|")

for row in history:
    n = row["n"]
    p = row["p_n"]
    step = row["|p_n - p_(n-1)|"]
    residual = row["|f(p_n)|"]

    step_str = "-" if math.isnan(step) else f"{step:.10e}"
    print(f"{n:2d}  {p: .10f}   {step_str:>14}   {residual:.10e}")

Approximate root: 0.824132312322
f(root): 4.648e-11

Iteration errors:
n   p_n            |p_n-p_(n-1)|    |f(p_n)|
 1   0.5000000000                -   6.2758256189e-01
 2   0.7500000000   2.5000000000e-01   1.6918886887e-01
 3   0.8750000000   1.2500000000e-01   1.2462814184e-01
 4   0.8125000000   6.2500000000e-02   2.7529312221e-02
 5   0.8437500000   3.1250000000e-02   4.7248334906e-02
 6   0.8281250000   1.5625000000e-02   9.5328213483e-03
 7   0.8203125000   7.8125000000e-03   9.0800932517e-03
 8   0.8242187500   3.9062500000e-03   2.0592391036e-04
 9   0.8222656250   1.9531250000e-03   4.4421974384e-03
10   0.8232421875   9.7656250000e-04   2.1194146147e-03
11   0.8237304688   4.8828125000e-04   9.5706477212e-04
12   0.8239746094   2.4414062500e-04   3.7565028053e-04
13   0.8240966797   1.2207031250e-04   8.4883146832e-05
14   0.8241577148   6.1035156250e-05   6.0515391411e-05
15   0.8241271973   3.0517578125e-05   1.2185125309e-05
16   0.8241424561   1.5258789062e-05   2.41648

In [ ]:
def fixed_point_iteration(
    g,
    p0: float,
    tol: float = 1e-10,
    max_iter: int = 100,
):
    """Fixed-point iteration: p_{n+1} = g(p_n). Return root and per-iteration data."""
    rows = []

    for n in range(max_iter):
        p_new = g(p0)
        step_error = abs(p_new - p0)

        rows.append(
            {
                "n": n + 1,
                "p_n": p_new,
                "|p_n - p_(n-1)|": step_error,
            }
        )

        if step_error < tol:
            return p_new, rows

        p0 = p_new

    return p0, rows


# Define g_1(x) = x + f(x)
g1 = lambda x: x + f(x)

# Check derivative: g_1'(x) = 1 + f'(x) = 1 + (3x^2 + 1)
f_prime = lambda x: 3 * x**2 + 1
g1_prime = lambda x: 1 + f_prime(x)

print("Fixed-point iteration with g_1(x) = x + f(x):")
print(f"g_1'(0) = {g1_prime(0):.4f}")
print(f"g_1'(1) = {g1_prime(1):.4f}")
print(f"g_1'(root) ≈ {g1_prime(root):.4f}")
print("\nSince |g_1'(x)| > 1 for all x in [0,1], convergence is NOT guaranteed.\n")

try:
    root_fpi, history_fpi = fixed_point_iteration(g1, p0=0.5, tol=1e-10, max_iter=100)
    print(f"Converged to: {root_fpi:.12f}")
    print(f"f(root): {f(root_fpi):.3e}")
    print("\nIteration errors:")
    print("n   p_n            |p_n - p_(n-1)|")
    for row in history_fpi:
        n = row["n"]
        p = row["p_n"]
        step = row["|p_n - p_(n-1)|"]
        print(f"{n:3d}  {p: .10f}   {step:.10e}")
except Exception as e:
    print(f"Error: {e}")

Fixed-point iteration with g_1(x) = x + f(x):
g_1'(0) = 2.0000
g_1'(1) = 5.0000
g_1'(root) ≈ 4.0376

Since |g_1'(x)| > 1 for all x in [0,1], convergence is NOT guaranteed.

Converged to: -1.151552290071
f(root): 9.190e-01

Iteration errors:
n   p_n            |p_n - p_(n-1)|
  1  -0.1275825619   6.2758256189e-01
  2  -1.1031776304   9.7559506848e-01
  3  -0.3369386548   7.6623897559e-01
  4  -1.1671821631   8.3024350830e-01
  5  -0.1976126182   9.6956954484e-01
  6  -1.1390999553   9.4148733705e-01
  7  -0.2599633926   8.7913656266e-01
  8  -1.1587818157   8.9881842303e-01
  9  -0.2164625874   9.4231922826e-01
 10  -1.1462698458   9.2980725836e-01
 11  -0.2442246279   9.0204521787e-01
 12  -1.1549040638   9.1067943587e-01
 13  -0.2251069615   9.2979710224e-01
 14  -1.1492040551   9.2409709358e-01
 15  -0.2377479161   9.1145613901e-01
 16  -1.1530946819   9.1534676581e-01
 17  -0.2291281118   9.2396657009e-01
 18  -1.1504932160   9.2136510419e-01
 19  -0.2348957774   9.1559743859e-01
 2

## Fixed-Point Iteration

A fixed point of a function $g$ is a value $p$ such that $g(p) = p$. If $f(p) = 0$, then $p$ is also a fixed point of $g_1(x) = x + f(x)$.

We can solve for roots by iterating $p_{n+1} = g_1(p_n)$ until convergence.

**Convergence condition:** Fixed-point iteration converges if $|g'(p)| < 1$ near the root.

In [ ]:
# Define g_2(x) = x - x*f(x)
g2 = lambda x: x - x * f(x)

# Derivative: g_2'(x) = 1 - f(x) - x*f'(x)
g2_prime = lambda x: 1 - f(x) - x * f_prime(x)

print("Fixed-point iteration with g_2(x) = x - x·f(x):")
print(f"g_2'(0) = {g2_prime(0):.6f}")
print(f"g_2'(1) = {g2_prime(1):.6f}")
print(f"g_2'(root) ≈ {g2_prime(root):.6f}")
print(f"\nSince |g_2'(root)| < 1, convergence IS guaranteed.\n")

root_fpi_g2, history_fpi_g2 = fixed_point_iteration(g2, p0=0.5, tol=1e-10, max_iter=100)
print(f"Converged to: {root_fpi_g2:.12f}")
print(f"f(root): {f(root_fpi_g2):.3e}")
print("\nIteration errors:")
print("n   p_n            |p_n - p_(n-1)|")
for row in history_fpi_g2:
    n = row["n"]
    p = row["p_n"]
    step = row["|p_n - p_(n-1)|"]
    print(f"{n:3d}  {p: .10f}   {step:.10e}")

Fixed-point iteration with g_2(x) = x - x·f(x):
g_2'(0) = 2.000000
g_2'(1) = -3.459698
g_2'(root) ≈ -1.503370

Since |g_2'(root)| < 1, convergence IS guaranteed.

Converged to: 0.824132315569
f(root): 7.782e-09

Iteration errors:
n   p_n            |p_n - p_(n-1)|
  1   0.8137912809   3.1379128095e-01
  2   0.8337220540   1.9930773091e-02
  3   0.8145731324   1.9148921604e-02
  4   0.8330228761   1.8449743676e-02
  5   0.8152918949   1.7730981211e-02
  6   0.8323763944   1.7084499517e-02
  7   0.8159534235   1.6422970916e-02
  8   0.8317782378   1.5824814332e-02
  9   0.8165628840   1.5215353877e-02
 10   0.8312244846   1.4661600649e-02
 11   0.8171248600   1.4099624579e-02
 12   0.8307115990   1.3586739004e-02
 13   0.8176434400   1.3068159054e-02
 14   0.8302363799   1.2592939925e-02
 15   0.8181222868   1.2114093118e-02
 16   0.8297959181   1.1673631349e-02
 17   0.8185646957   1.1231222456e-02
 18   0.8293875616   1.0822865933e-02
 19   0.8189736417   1.0413919911e-02
 20   0.82900

## Convergent Fixed-Point: $g_2(x) = x - x \cdot f(x)$

A better choice is $g_2(x) = x - x \cdot f(x) = x(1 - f(x))$.

At a root $p$ where $f(p) = 0$:
$$g_2'(p) = 1 - f(p) - p \cdot f'(p) = 1 - p \cdot f'(p)$$

For $f(x) = x^3 + x - 1$, we have $f'(x) = 3x^2 + 1$. 

If $|g_2'(p)| < 1$, fixed-point iteration **will converge**.

In [9]:
# Define g_3(x) = x - f(x) / f'(x)  [Newton's method]
g3 = lambda x: x - f(x) / f_prime(x)

# Derivative: g_3'(x) = 1 - [f'(x)^2 - f(x)*f''(x)] / f'(x)^2
#                      = 1 - [f'(x)^2 - f(x)*f''(x)] / f'(x)^2
#                      = [f(x) * f''(x)] / f'(x)^2
f_double_prime = lambda x: 6 * x  # d/dx(3x^2 + 1) = 6x
g3_prime = lambda x: (f(x) * f_double_prime(x)) / (f_prime(x) ** 2)

print("Newton's method: g_3(x) = x - f(x)/f'(x):")
print(f"g_3'(0) = {g3_prime(0):.6f}")
print(f"g_3'(1) = {g3_prime(1):.6f}")
print(f"g_3'(root) ≈ {g3_prime(root):.6f}")
print(f"\nAt the root: g_3'(p) = 0, guaranteeing quadratic convergence.\n")

root_fpi_g3, history_fpi_g3 = fixed_point_iteration(g3, p0=0.5, tol=1e-11, max_iter=100)
print(f"Converged to: {root_fpi_g3:.15f}")
print(f"f(root): {f(root_fpi_g3):.3e}")
print("\nIteration errors:")
print("n   p_n                    |p_n - p_(n-1)|")
for row in history_fpi_g3:
    n = row["n"]
    p = row["p_n"]
    step = row["|p_n - p_(n-1)|"]
    print(f"{n:3d}  {p:.15f}   {step:.10e}")

Newton's method: g_3(x) = x - f(x)/f'(x):
g_3'(0) = -0.000000
g_3'(1) = 0.172387
g_3'(root) ≈ 0.000000

At the root: g_3'(p) = 0, guaranteeing quadratic convergence.

Converged to: 0.824132312303418
f(root): 2.133e-12

Iteration errors:
n   p_n                    |p_n - p_(n-1)|
  1  0.858618606794499   3.5861860679e-01
  2  0.832544325708388   2.6074281086e-02
  3  0.826006013477488   6.5383122309e-03
  4  0.824539493809837   1.4665196677e-03
  5  0.824220300289598   3.1919352024e-04
  6  0.824151302194038   6.8998095560e-05
  7  0.824136409676019   1.4892518019e-05
  8  0.824133196325543   3.2133504760e-06
  9  0.824132503031284   6.9329425878e-07
 10  0.824132353452314   1.4957897054e-07
 11  0.824132321180598   3.2271715966e-08
 12  0.824132314217969   6.9626292509e-09
 13  0.824132312715780   1.5021883737e-09
 14  0.824132312391683   3.2409741557e-10
 15  0.824132312321759   6.9924177559e-11
 16  0.824132312306673   1.5086043526e-11
 17  0.824132312303418   3.2548408413e-12


## Newton's Method: $g_3(x) = x - \frac{f(x)}{f'(x)}$

Newton's method is the gold standard of root finding when derivatives are available. It reformulates as fixed-point iteration with:

$$g_3(x) = x - \frac{f(x)}{f'(x)}$$

**Convergence:** At a simple root where $f(p) = 0$ and $f'(p) \ne 0$:
$$g_3'(p) = 0$$

So $|g_3'(p)| = 0 < 1$, guaranteeing **superlinear (quadratic) convergence** near the root.

In [10]:
h = lambda x: math.atan(x) - math.cos(x)
h_prime = lambda x: 1 / (1 + x**2) + math.sin(x)

# Run both methods
root_bis, hist_bis = bisection_with_errors(h, a=0.0, b=1.0, tol=1e-12, max_iter=100)
newton_g = lambda x: x - h(x) / h_prime(x)
root_newt, hist_newt = fixed_point_iteration(newton_g, p0=0.5, tol=1e-15, max_iter=50)

print(f"Bisection  root: {root_bis:.15f}  ({len(hist_bis)} iterations)")
print(f"Newton     root: {root_newt:.15f}  ({len(hist_newt)} iterations)")
print(f"Agreement: |bisection - newton| = {abs(root_bis - root_newt):.3e}")

# Side-by-side comparison table
max_rows = max(len(hist_bis), len(hist_newt))
print(f"\n{'n':>3}  {'Bisection |error|':>20}  {'Newton |error|':>20}")
print("-" * 50)
for i in range(max_rows):
    bis_err = f"{hist_bis[i]['|f(p_n)|']:.6e}" if i < len(hist_bis) else "converged"
    newt_err = f"{hist_newt[i]['|p_n - p_(n-1)|']:.6e}" if i < len(hist_newt) else "converged"
    print(f"{i+1:>3}  {bis_err:>20}  {newt_err:>20}")

Bisection  root: 0.816541226173285  (34 iterations)
Newton     root: 0.816541226172734  (5 iterations)
Agreement: |bisection - newton| = 5.516e-13

  n     Bisection |error|        Newton |error|
--------------------------------------------------
  1          4.139350e-01          3.235319e-01
  2          8.818776e-02          6.988901e-03
  3          7.783314e-02          1.755397e-06
  4          5.369007e-03          1.122435e-13
  5          3.618868e-02          0.000000e+00
  6          1.539843e-02             converged
  7          5.011787e-03             converged
  8          1.793495e-04             converged
  9          2.416035e-03             converged
 10          1.118297e-03             converged
 11          4.694621e-04             converged
 12          1.450535e-04             converged
 13          1.714873e-05             converged
 14          6.395218e-05             converged
 15          2.340168e-05             converged
 16          3.126468e-06        

## Comparison: Bisection vs Newton on $f(x) = \arctan(x) - \cos(x)$

We compare the two methods on a new function:

$$f(x) = \arctan(x) - \cos(x), \quad x \in [0, 1]$$

with derivative $f'(x) = \dfrac{1}{1+x^2} + \sin(x)$.

Note: $f(0) = -1 < 0$ and $f(1) = \arctan(1) - \cos(1) > 0$, so the sign-change condition holds.

We expect bisection to converge **linearly** (halving the error each step) while Newton converges **quadratically** (doubling the number of correct digits each step).

In [11]:
phi = lambda x: abs(math.sin(x)) * (x - math.pi)
phi_prime = lambda x: (1 if math.sin(x) >= 0 else -1) * math.cos(x) * (x - math.pi) + abs(math.sin(x))

print(f"f(3)   = {phi(3):.6f}  (< 0 ✓)")
print(f"f(4)   = {phi(4):.6f}  (> 0 ✓)")
print(f"f'(π)  = {phi_prime(math.pi):.2e}  (≈ 0: double root)")

# Bisection on [3, 4]
root_phi_bis, hist_phi_bis = bisection_with_errors(phi, a=3.0, b=4.0, tol=1e-12, max_iter=200)

# Newton starting at x₀ = 3
newton_phi = lambda x: x - phi(x) / phi_prime(x)
root_phi_newt, hist_phi_newt = fixed_point_iteration(newton_phi, p0=3.0, tol=1e-12, max_iter=200)

print(f"\nBisection root: {root_phi_bis:.15f}  ({len(hist_phi_bis)} iterations)")
print(f"Newton    root: {root_phi_newt:.15f}  ({len(hist_phi_newt)} iterations)")
print(f"True root (π):  {math.pi:.15f}")
print(f"\nBisection error: {abs(root_phi_bis - math.pi):.3e}")
print(f"Newton    error: {abs(root_phi_newt - math.pi):.3e}")

# Side-by-side error table (first 40 rows)
max_rows = min(max(len(hist_phi_bis), len(hist_phi_newt)), 40)
print(f"\n{'n':>3}  {'Bisection |f(pₙ)|':>22}  {'Newton |step|':>22}")
print("-" * 54)
for i in range(max_rows):
    bis_e = f"{hist_phi_bis[i]['|f(p_n)|']:.6e}" if i < len(hist_phi_bis) else "converged"
    nwt_e = f"{hist_phi_newt[i]['|p_n - p_(n-1)|']:.6e}" if i < len(hist_phi_newt) else "converged"
    print(f"{i+1:>3}  {bis_e:>22}  {nwt_e:>22}")

f(3)   = -0.019982  (< 0 ✓)
f(4)   = 0.649645  (> 0 ✓)
f'(π)  = 1.22e-16  (≈ 0: double root)

Bisection root: 3.141592025756836  (19 iterations)
Newton    root: 3.141592653589281  (38 iterations)
True root (π):  3.141592653589793

Bisection error: 6.278e-07
Newton    error: 5.125e-13

  n       Bisection |f(pₙ)|           Newton |step|
------------------------------------------------------
  1            1.257233e-01            7.103400e-02
  2            1.172915e-02            3.530863e-02
  3            2.753035e-04            1.762866e-02
  4            2.106744e-03            8.811136e-03
  5            2.148301e-04            4.405169e-03
  6            9.363533e-07            2.202535e-03
  7            4.685156e-05            1.101261e-03
  8            8.635336e-06            5.506297e-04
  9            9.711537e-07            2.753148e-04
 10            7.936868e-11            1.376574e-04
 11            2.297978e-07            6.882868e-05
 12            5.533396e-08        

## A Nasty Root: $f(x) = |\sin(x)|(x - \pi)$

Consider $f(x) = |\sin(x)|\,(x-\pi)$. The unique root in $[3, 4]$ is $x = \pi$.

Near the root, $|\sin(x)| \approx |x - \pi|$, so:
$$f(x) \approx |x-\pi|\,(x-\pi) = \text{sign}(x-\pi)\,(x-\pi)^2$$

This is essentially a **double root**: $f(\pi) = 0$ and $f'(\pi) = 0$.

$$f'(x) = \text{sign}(\sin x)\cos(x)\,(x-\pi) + |\sin x|$$

At $x = \pi$: both terms vanish, so $f'(\pi) = 0$. Newton's formula $x - f(x)/f'(x)$ divides by zero at the root — the method still converges, but only **linearly**, not quadratically.

In [12]:
def secant_method(
    f,
    x0: float,
    x1: float,
    tol: float = 1e-12,
    max_iter: int = 100,
):
    """Secant method: x_{n+1} = x_n - f(x_n) * (x_n - x_{n-1}) / (f(x_n) - f(x_{n-1}))"""
    rows = []
    f0, f1 = f(x0), f(x1)

    for n in range(max_iter):
        if abs(f1 - f0) < 1e-16:
            break
        x_new = x1 - f1 * (x1 - x0) / (f1 - f0)
        f_new = f(x_new)
        
        step_error = abs(x_new - x1)
        rows.append(
            {
                "n": n + 1,
                "x_n": x_new,
                "|x_n - x_{n-1}|": step_error,
            }
        )
        
        if step_error < tol:
            return x_new, rows
        
        x0, f0 = x1, f1
        x1, f1 = x_new, f_new

    return x1, rows


# Use h(x) = arctan(x) - cos(x) from earlier
# Newton's method starting at x=0.5
newton_sec = lambda x: x - h(x) / h_prime(x)
root_newton_sec, hist_newton_sec = fixed_point_iteration(newton_sec, p0=0.5, tol=1e-14, max_iter=50)

# Secant method starting at x₀=0.0, x₁=1.0
root_secant, hist_secant = secant_method(h, x0=0.0, x1=1.0, tol=1e-14, max_iter=50)

print("Newton vs Secant on f(x) = arctan(x) - cos(x):\n")
print(f"Newton  root: {root_newton_sec:.15f}  ({len(hist_newton_sec)} iterations)")
print(f"Secant  root: {root_secant:.15f}  ({len(hist_secant)} iterations)")
print(f"Difference:   {abs(root_newton_sec - root_secant):.3e}\n")

# Side-by-side error table
max_iter_display = max(len(hist_newton_sec), len(hist_secant))
print(f"{'n':>3}  {'Newton |step|':>20}  {'Secant |step|':>20}")
print("-" * 50)
for i in range(max_iter_display):
    newton_e = f"{hist_newton_sec[i]['|p_n - p_(n-1)|']:.6e}" if i < len(hist_newton_sec) else "converged"
    secant_e = f"{hist_secant[i]['|x_n - x_{n-1}|']:.6e}" if i < len(hist_secant) else "converged"
    print(f"{i+1:>3}  {newton_e:>20}  {secant_e:>20}")

Newton vs Secant on f(x) = arctan(x) - cos(x):

Newton  root: 0.816541226172734  (5 iterations)
Secant  root: 0.816541226172734  (6 iterations)
Difference:   0.000e+00

  n         Newton |step|         Secant |step|
--------------------------------------------------
  1          3.235319e-01          1.968490e-01
  2          6.988901e-03          1.331663e-02
  3          1.755397e-06          7.362092e-05
  4          1.122435e-13          3.643556e-08
  5          0.000000e+00          9.781065e-14
  6             converged          0.000000e+00


## The Secant Method: Derivative-Free Alternative

The **secant method** approximates the derivative using finite differences, avoiding the need to compute $f'(x)$:

$$x_{n+1} = x_n - f(x_n) \frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}$$

**Advantages:**
- No derivative required (useful when $f'$ is expensive or undefined)
- Requires only two starting points $x_0, x_1$

**Convergence:**
- **Superlinear** with order $\phi = \frac{1 + \sqrt{5}}{2} \approx 1.618$ (the golden ratio!)
- Faster than bisection/linear methods, but slower than Newton's quadratic convergence

**Comparison on $f(x) = \arctan(x) - \cos(x)$ on $[0,1]$:**

In [13]:
def false_position(
    f,
    a: float,
    b: float,
    tol: float = 1e-12,
    max_iter: int = 100,
):
    """False position method: x_n = a - f(a)*(b-a)/(f(b)-f(a))"""
    fa, fb = f(a), f(b)
    if fa * fb >= 0:
        raise ValueError("f(a) and f(b) must have opposite signs.")
    
    rows = []
    
    for n in range(max_iter):
        # Linear interpolation estimate
        x_fp = a - fa * (b - a) / (fb - fa)
        fx_fp = f(x_fp)
        
        step_error = abs(x_fp - a) if n == 0 else abs(x_fp - rows[-1]["x_n"])
        rows.append(
            {
                "n": n + 1,
                "x_n": x_fp,
                "|x_n - x_{n-1}|": step_error,
            }
        )
        
        if abs(fx_fp) < tol or step_error < tol:
            return x_fp, rows
        
        # Update bracket
        if fa * fx_fp < 0:
            b, fb = x_fp, fx_fp
        else:
            a, fa = x_fp, fx_fp
    
    return a, rows


# Three-way comparison on h(x) = arctan(x) - cos(x)
# 1. Newton (from before)
# 2. Secant (from before) 
# 3. False Position
root_fp, hist_fp = false_position(h, a=0.0, b=1.0, tol=1e-14, max_iter=100)

print("Three-way comparison on f(x) = arctan(x) - cos(x):\n")
print(f"Newton       root: {root_newton_sec:.15f}  ({len(hist_newton_sec)} iterations)")
print(f"Secant       root: {root_secant:.15f}  ({len(hist_secant)} iterations)")
print(f"False Position root: {root_fp:.15f}  ({len(hist_fp)} iterations)")
print(f"\nAll methods converge to: {root_newton_sec:.15f}\n")

# Side-by-side error table
max_iter_all = max(len(hist_newton_sec), len(hist_secant), len(hist_fp))
print(f"{'n':>3}  {'Newton |step|':>18}  {'Secant |step|':>18}  {'FalsePos |step|':>18}")
print("-" * 66)
for i in range(max_iter_all):
    newton_e = f"{hist_newton_sec[i]['|p_n - p_(n-1)|']:.5e}" if i < len(hist_newton_sec) else "conv."
    secant_e = f"{hist_secant[i]['|x_n - x_{n-1}|']:.5e}" if i < len(hist_secant) else "conv."
    fp_e = f"{hist_fp[i]['|x_n - x_{n-1}|']:.5e}" if i < len(hist_fp) else "conv."
    print(f"{i+1:>3}  {newton_e:>18}  {secant_e:>18}  {fp_e:>18}")

Three-way comparison on f(x) = arctan(x) - cos(x):

Newton       root: 0.816541226172734  (5 iterations)
Secant       root: 0.816541226172734  (6 iterations)
False Position root: 0.816541226172734  (7 iterations)

All methods converge to: 0.816541226172734

  n       Newton |step|       Secant |step|     FalsePos |step|
------------------------------------------------------------------
  1         3.23532e-01         1.96849e-01         8.03151e-01
  2         6.98890e-03         1.33166e-02         1.33166e-02
  3         1.75540e-06         7.36209e-05         7.31869e-05
  4         1.12244e-13         3.64356e-08         3.95388e-07
  5         0.00000e+00         9.78106e-14         2.13586e-09
  6               conv.         0.00000e+00         1.15379e-11
  7               conv.               conv.         6.22835e-14


## False Position Method (Regula Falsi)

The **false position method** is a bracketing method that replaces bisection's midpoint with a weighted average based on function values:

$$x_n = a - f(a) \frac{b - a}{f(b) - f(a)}$$

where $[a, b]$ brackets the root.

**Advantages:**
- Always brackets the root (like bisection) — guaranteed convergence
- Uses function information to accelerate convergence
- No derivative required

**Convergence:**
- **Linear** like bisection, but often **faster** in practice
- May suffer from one-sided convergence if $f''$ doesn't change sign

**Full comparison:** Newton (quadratic) vs Secant (order $\phi \approx 1.618$) vs False Position (linear but informed)